In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .config("spark.sql.warehouse.dir","spark-warehouse") \
    .enableHiveSupport() \
    .getOrCreate()


In [4]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", "true")

In [5]:
spark.sql("""
CREATE TABLE dim_products
USING PARQUET
LOCATION 'file:/home/jovyan/spark-warehouse/dim_products'
""")

spark.sql("""
CREATE TABLE dim_customers
USING PARQUET
LOCATION 'file:/home/jovyan/spark-warehouse/dim_customers'
""")

spark.sql("""
CREATE TABLE fact_order_items
USING PARQUET
LOCATION 'file:/home/jovyan/spark-warehouse/fact_order_items'
""")

spark.sql("""
CREATE TABLE dim_sellers
USING PARQUET
LOCATION 'file:/home/jovyan/spark-warehouse/dim_sellers'
""")

""


In [8]:
fact_order_item = spark.table("fact_order_items")
fact_order_item.groupBy("customer_key").count().count()

12423

In [9]:
dim_customers = spark.table("dim_customers")
dim_customers.groupBy("customer_key").count().count()

12422

In [11]:
fact_order_item.select("customer_key").distinct().join(dim_customers.select("customer_key"), on=['customer_key']).count()

12246

In [13]:
dim_sellers = spark.table("dim_sellers")
dim_sellers.groupBy("seller_key").count().count()

3095

In [14]:
fact_order_item.groupBy("seller_key").count().count()

1628

In [15]:
fact_order_item.select("seller_key").distinct().join(dim_sellers.select("seller_key"), on=['seller_key']).count()

1628

In [16]:
dim_products = spark.table("dim_products")
dim_products.groupBy("product_key").count().count()

32951

In [17]:
fact_order_item.groupBy("product_key").count().count()

7474

In [19]:
fact_order_item.select("product_key").distinct().join(dim_products.select("product_key"), on=['product_key']).count()

7474